In [1]:
# ==========================================================
# Notebook 10: End-to-End Personal Knowledge Engine
# ==========================================================

import faiss
import pickle
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

import torch

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
# Load corpus
chunks_df = pd.read_csv("data/processed/chunked_corpus.csv")

# Load embeddings
with open("data/embeddings/chunk_embeddings.pkl", "rb") as file:

    embeddings = pickle.load(file)

embeddings = np.array(embeddings).astype("float32")

# Load FAISS index
faiss_index = faiss.read_index("data/embeddings/faiss_index.bin")

print("Artifacts loaded successfully.")

Artifacts loaded successfully.


In [3]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: f66df7d2-e68b-4d3f-9712-755101ea2c67)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json
Retrying in 1s [Retry 1/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: e7e69eea-79c0-4b01-a8ed-d70dcaa22ea5)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json
Retrying in 2s [Retry 2/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: a1be50c8-f92d-49e0-ba6c-858926826748)')' thrown while requesting HEAD https://huggingface.co/api/resolve-cache/models/sentence-

In [4]:
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", torch_dtype=torch.float16, trust_remote_code=True
)

generator = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.1,
    do_sample=False,
    return_full_text=False,
)

print("Phi-3 Mini Loaded.")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
`flash-attention` package not found, consider installing for better performance: No module named 'flash_attn'.
Current `flash-attention` does not support `window_size`. Either upgrade or use `attn_implementation='eager'`.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device device because they were offloaded to the disk and cpu.


Phi-3 Mini Loaded.


In [5]:
def retrieve_documents(query, top_k=5):

    query_embedding = embedding_model.encode(query)

    query_embedding = np.array([query_embedding]).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = faiss_index.search(query_embedding, top_k)

    results = chunks_df.iloc[indices[0]].copy()

    results["score"] = scores[0]

    return results

In [6]:
def build_prompt(query, retrieved_docs):

    context = "\n\n".join(retrieved_docs["chunk_text"].tolist())

    return f"""
<|system|>
You are a helpful personal AI assistant.
Answer only from the provided context.
If the answer is unavailable, clearly state that.

<|user|>

Context:
{context}

Question:
{query}

<|assistant|>
"""

In [7]:
def local_llm(prompt):

    output = generator(prompt)

    return output[0]["generated_text"].strip()

In [8]:
def personal_knowledge_engine(query, top_k=5):

    retrieved_docs = retrieve_documents(query=query, top_k=top_k)

    prompt = build_prompt(query=query, retrieved_docs=retrieved_docs)

    answer = local_llm(prompt)

    return {
        "query": query,
        "retrieved_docs": retrieved_docs,
        "prompt": prompt,
        "answer": answer,
    }

In [11]:
result = personal_knowledge_engine(query="How does hybrid search work?", top_k=3)

print("=" * 80)
print("QUESTION:")
print(result["query"])

print("\n" + "=" * 80)
print("RETRIEVED SOURCES:\n")

print(result["retrieved_docs"][["source", "score"]])

print("\n" + "=" * 80)
print("ANSWER:\n")

print(result["answer"])

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\transformers\generation\configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
You are not running the flash-attention implementation, expect numerical differences.


QUESTION:
How does hybrid search work?

RETRIEVED SOURCES:

        source     score
1388  book.pdf  0.513751
1666  book.pdf  0.484536
2028  book.pdf  0.437373

ANSWER:

Hybrid search combines the strengths of both traditional keyword-based search and vector-based search to provide more accurate and relevant results. In a hybrid search system, the search process involves two main steps:

1. Keyword-based search: The system first performs a traditional keyword-based search using the user's query. This step involves searching through the indexed documents using a Boolean test to filter out the candidate documents that match the query terms. The search results are ranked based on Lucene's practical scoring function, which considers factors such as term frequency, document length, and document relevance.

2. Vector-based search: In the second step, the system performs a vector-based search to further refine the search results. This involves using a vector representation of the documents an